In [ ]:
import os

print(os.getcwd())
print(os.listdir("../data/raw"))

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/survey_responses.csv")

df.head()

df.head()

In [ ]:
df.columns


In [ ]:
df.isnull().sum()

In [ ]:
df.columns = [
    "timestamp",
    "name",
    "budget",
    "food",
    "cafes",
    "entertainment",
    "arts",
    "sports",
    "outdoor",
    "shopping",
    "nightlife",
    "transportation",
    "max_travel_time",
    "important_factors"
]

In [ ]:
df.columns

In [ ]:
print(df["budget"].unique())
print(df["transportation"].unique())
print(df["max_travel_time"].unique())
print(df["important_factors"].unique())

In [ ]:
df["budget_raw"] = df["budget"]
df["budget"] = df["budget"].str.replace("$", "", regex=False)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
df[["budget_raw", "budget"]]

In [ ]:
df["budget"].describe()

In [ ]:
travel_time_map = {
    "≤ 10 minutes": 10,
    "≤ 20 minutes": 20,
    "≤ 30 minutes": 30,
    "≤ 45 minutes": 45,
    "≤ 60 minutes": 60
}
df["max_travel_minutes"] = df["max_travel_time"].map(travel_time_map)
df[["max_travel_time", "max_travel_minutes"]]
df["max_travel_minutes"].isnull().sum()

In [ ]:
transport = df["transportation"].str.get_dummies(sep=", ")
transport = transport.rename(columns={
    "Walking": "transport_walking",
    "Public transportation": "transport_public",
    "Bike/scooter": "transport_bike",
    "Car": "transport_car",
    "Rideshare": "transport_rideshare"
})
transport.head()
df = pd.concat([df, transport], axis=1)
df.head()
df[[
    "name",
    "transport_walking",
    "transport_public",
    "transport_bike",
    "transport_car",
    "transport_rideshare"
]].head()

In [ ]:

df["important_factors"] = df["important_factors"].str.replace(
    "Something new/",
    "Something new",
    regex=False
)
factors = df["important_factors"].str.get_dummies(sep=", ")
factors.columns
factors.sum().sort_values(ascending=False)
factors = factors.rename(columns={
    "Everyone likes it": "factor_group_agreement",
    "Something new": "factor_novelty",
    "Safety": "factor_safety",
    "Food options": "factor_food",
    "Convenient transportation": "factor_transport",
    "Short travel time": "factor_travel_time",
    "Low cost": "factor_cost",
    "High ratings": "factor_rating",
    "Match my interests": "factor_interest"
})
df = pd.concat([df, factors], axis=1)
df.shape


In [ ]:
interest_columns = [
    "food",
    "cafes",
    "entertainment",
    "arts",
    "sports",
    "outdoor",
    "shopping",
    "nightlife"
]
df[interest_columns].describe()
average_interests = df[interest_columns].mean()
average_interests.sort_values(ascending=False)

In [ ]:
df["name"].tolist()

In [ ]:
group_names = [
    "Thai Bao Tram",
    "Ngô Thu Ngân",
    "Nguyen Van Trung "
]

group = df[df["name"].isin(group_names)]
group[["name"] + interest_columns]

In [ ]:
print(df.columns.tolist())

In [ ]:
group_average = group[interest_columns].mean()

group_average

In [ ]:
group_disagreement = group[interest_columns].std()

group_disagreement

In [ ]:
fair_scores = group_average - 0.5 * group_disagreement

fair_scores.sort_values(ascending=False)

In [ ]:
group_minimum = group[interest_columns].min()

group_minimum.sort_values(ascending=False)

In [ ]:
import sys

sys.path.append("../src")

from recommender import calculate_group_scores
scores = calculate_group_scores(group, interest_columns)

scores

In [ ]:
places = pd.read_csv("../data/processed/places.csv")

places["interest_score"] = places["category"].map(scores["fair_score"])
places

In [ ]:
group_budget = group["budget"].min()
places["within_budget"] = places["price"] <= group_budget
places[[
    "name",
    "category",
    "price",
    "interest_score",
    "within_budget"
]]

In [ ]:
places["place_score"] = (
    0.7 * places["interest_score"]
    + 0.3 * places["rating"]
)
eligible_places = places[places["within_budget"]].copy()
ranked_places = eligible_places.sort_values(
    "place_score",
    ascending=False
)

ranked_places[[
    "name",
    "category",
    "price",
    "rating",
    "interest_score",
    "place_score"
]]

In [ ]:
places["travel_person1"] = [8, 15, 20, 25, 30, 18, 12, 35]
places["travel_person2"] = [12, 20, 15, 18, 25, 22, 16, 30]
places["travel_person3"] = [10, 25, 18, 30, 20, 15, 14, 28]
places[[
    "name",
    "travel_person1",
    "travel_person2",
    "travel_person3"
]]
travel_columns = [
    "travel_person1",
    "travel_person2",
    "travel_person3"
]

places["avg_travel_time"] = places[travel_columns].mean(axis=1)
places["max_travel_time"] = places[travel_columns].max(axis=1)
places[[
    "name",
    "avg_travel_time",
    "max_travel_time"
]]
group[["name", "max_travel_minutes"]]
travel_limits = group["max_travel_minutes"].tolist()

travel_limits
places["person1_travel_ok"] = (
    places["travel_person1"] <= travel_limits[0]
)

places["person2_travel_ok"] = (
    places["travel_person2"] <= travel_limits[1]
)

places["person3_travel_ok"] = (
    places["travel_person3"] <= travel_limits[2]
)
places["travel_ok"] = (
    places["person1_travel_ok"]
    & places["person2_travel_ok"]
    & places["person3_travel_ok"]
)
places[[
    "name",
    "avg_travel_time",
    "max_travel_time",
    "person1_travel_ok",
    "person2_travel_ok",
    "person3_travel_ok",
    "travel_ok"
]]

In [ ]:
def check_travel_constraints(places, group, travel_columns):
    travel_limits = group["max_travel_minutes"].tolist()

    if len(travel_limits) != len(travel_columns):
        raise ValueError("Number of group members must match travel columns")

    travel_ok = pd.Series(True, index=places.index)

    for i in range(len(travel_columns)):
        travel_ok = travel_ok & (
            places[travel_columns[i]] <= travel_limits[i]
        )

    return travel_ok

In [ ]:
from recommender import (
    calculate_group_scores,
    check_travel_constraints
)
places["travel_ok"] = check_travel_constraints(
    places,
    group,
    travel_columns
)

places[[
    "name",
    "avg_travel_time",
    "max_travel_time",
    "travel_ok"
]]

In [ ]:
import sys
import importlib

sys.path.append("../src")

import recommender
importlib.reload(recommender)

from recommender import (
    calculate_group_scores,
    check_travel_constraints
)

In [ ]:
import importlib
import recommender

importlib.reload(recommender)

from recommender import (
    calculate_group_scores,
    check_travel_constraints,
    rank_places
)
recommendations = rank_places(
    places,
    group,
    scores,
    travel_columns
)
recommendations[[
    "name",
    "category",
    "price",
    "rating",
    "interest_score",
    "avg_travel_time",
    "max_travel_time",
    "place_score"
]].head(3)

In [ ]:
import importlib
import recommender

importlib.reload(recommender)

In [ ]:
recommendations = recommender.rank_places(
    places,
    group,
    scores,
    travel_columns
)

In [ ]:
recommendations[[
    "name",
    "category",
    "price",
    "rating",
    "interest_score",
    "avg_travel_time",
    "travel_score",
    "place_score"
]].head(3)

In [ ]:
travel_times = pd.read_csv(
    "../data/processed/travel_times.csv"
)
group_travel = travel_times[
    travel_times["user_id"].isin(group["user_id"])
].copy()
travel_times.head()

In [ ]:
group = group.copy()

group["user_id"] = [
    f"user_{i+1:03d}"
    for i in range(len(group))
]

group[["name", "user_id", "max_travel_minutes"]]

In [ ]:
travel_check = group_travel.merge(
    group[["user_id", "max_travel_minutes"]],
    on="user_id",
    how="left"
)

travel_check.head(10)

In [ ]:
travel_check["travel_ok"] = (
    travel_check["travel_minutes"]
    <= travel_check["max_travel_minutes"]
)

travel_check.head(10)

In [ ]:
place_travel_ok = (
    travel_check.groupby("place")["travel_ok"]
    .all()
    .reset_index()
)

place_travel_ok

In [ ]:
places = pd.read_csv("../data/processed/places.csv")

places = places.merge(
    place_travel_ok,
    left_on="name",
    right_on="place",
    how="left"
)

places["travel_ok"] = places["travel_ok"].fillna(False)

places[["name", "category", "price", "travel_ok"]]

In [ ]:
import importlib
import recommender

importlib.reload(recommender)

scores = recommender.calculate_group_scores(
    group, interest_columns
)

recommendations = recommender.rank_places(
    places,
    group,
    scores,
    travel_times
)

recommendations[
    ["name", "category", "price",
     "interest_score", "travel_score",
     "place_score"]
].head(3)

In [ ]:
travel_results = recommender.check_travel_constraints(
    travel_times, group
)

budget_limit = group["budget"].min()

debug_places = places[["name", "price"]].copy()

debug_places["within_budget"] = (
    debug_places["price"] <= budget_limit
)

debug_places = debug_places.merge(
    travel_results,
    left_on="name",
    right_on="place",
    how="left"
)

debug_places["travel_ok"] = (
    debug_places["travel_ok"].fillna(False)
)

debug_places[
    ["name", "price", "within_budget", "travel_ok"]
]

In [ ]:
travel_check["travel_ok"] = (
    travel_check["travel_minutes"]
    <= travel_check["max_travel_minutes"]
)

travel_check[
    ["place", "user_id", "travel_minutes",
     "max_travel_minutes", "travel_ok"]
].sort_values(["place", "user_id"])

In [ ]:
# Step 1: Calculate extra travel minutes
travel_check["extra_minutes"] = (
    travel_check["travel_minutes"]
    - travel_check["max_travel_minutes"]
).clip(lower=0)

# Step 2: Find the maximum violation for each place
travel_violations = (
    travel_check.groupby("place")["extra_minutes"]
    .max()
    .reset_index()
)

# Step 3: Rename the column
travel_violations = travel_violations.rename(
    columns={"extra_minutes": "max_extra_minutes"}
)

# Step 4: Display results
travel_violations.sort_values("max_extra_minutes")

In [ ]:
group_budget = group["budget"].min()

fallback = places[["name", "category", "price", "rating"]].copy()

fallback["extra_cost"] = (
    fallback["price"] - group_budget
).clip(lower=0)

fallback = fallback.merge(
    travel_violations,
    left_on="name",
    right_on="place",
    how="left"
)

fallback[
    ["name", "price", "extra_cost", "max_extra_minutes"]
]

In [ ]:
# Calculate normalized constraint violations
fallback["budget_penalty"] = (
    fallback["extra_cost"] / max(group_budget, 1)
)

fallback["travel_penalty"] = (
    fallback["max_extra_minutes"]
    / max(group["max_travel_minutes"].mean(), 1)
)

# Combine penalties
fallback["total_penalty"] = (
    0.5 * fallback["budget_penalty"]
    + 0.5 * fallback["travel_penalty"]
)

# Exclude places already recommended
fallback_options = fallback[
    ~fallback["name"].isin(recommendations["name"])
].copy()

# Rank alternatives
fallback_options = fallback_options.sort_values(
    "total_penalty",
    ascending=True
)

fallback_options[
    ["name", "extra_cost", "max_extra_minutes", "total_penalty"]
].head(3)

In [ ]:
def explain_alternative(row):
    reasons = []

    if row["extra_cost"] > 0:
        reasons.append(
            f"${row['extra_cost']:.2f} over the group budget"
        )

    if row["max_extra_minutes"] > 0:
        reasons.append(
            f"up to {row['max_extra_minutes']:.0f} minutes "
            "beyond a member's travel limit"
        )

    if not reasons:
        return "Meets all budget and travel constraints"

    return "Requires flexibility: " + " and ".join(reasons)


fallback_options["explanation"] = fallback_options.apply(
    explain_alternative,
    axis=1
)

fallback_options[
    ["name", "extra_cost", "max_extra_minutes", "explanation"]
].head(3)
fallback_options.apply(explain_alternative, axis=1)

In [ ]:
# Fully eligible recommendations
eligible = recommendations.head(3).copy()

eligible["status"] = "Recommended"
eligible["explanation"] = (
    "Matches group preferences and meets all constraints"
)

# Fallback alternatives
remaining_slots = max(0, 3 - len(eligible))

alternatives = fallback_options.head(remaining_slots).copy()

alternatives["status"] = "Alternative"

# Combine the results
final_recommendations = pd.concat(
    [
        eligible[["name", "status", "explanation"]],
        alternatives[["name", "status", "explanation"]]
    ],
    ignore_index=True
)

final_recommendations

In [ ]:
# Create a test group without changing the original
test_group = group.copy()

# Increase user_003's travel limit to 20 minutes
test_group.loc[
    test_group["user_id"] == "user_003",
    "max_travel_minutes"
] = 20

# Calculate recommendations for the new group
test_scores = recommender.calculate_group_scores(
    test_group,
    interest_columns
)

test_recommendations = recommender.rank_places(
    places,
    test_group,
    test_scores,
    travel_times
)

test_recommendations[
    ["name", "price", "avg_travel_time", "place_score"]
]

In [ ]:
# Make a copy so we don't change the original data
test_travel = travel_times.copy()

# Remove user_003's travel record for City Park
test_travel = test_travel[
    ~(
        (test_travel["place"] == "City Park")
        & (test_travel["user_id"] == "user_003")
    )
]

# Check travel eligibility
missing_data_test = recommender.check_travel_constraints(
    test_travel,
    test_group
)

missing_data_test[
    missing_data_test["place"] == "City Park"
]

In [ ]:
group = group.copy()

group["user_id"] = [
    f"user_{i+1:03d}"
    for i in range(len(group))
]

print(group[["name", "user_id", "max_travel_minutes"]])